<div align="center">

# Trivia Night: KONASH Agent vs. Claude Opus 4.6

**A $0.10/query search agent takes on the world's most powerful AI — head to head**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/konaequity/konash/blob/main/notebooks/trivia_night.ipynb)

</div>

---

**The bet:** A search-augmented agent built on GLM 4.5 Air can beat Claude Opus 4.6 on hard trivia — because finding the right passage beats having a bigger brain. This is the core thesis of the [KARL paper](https://arxiv.org/abs/2503.xxxxx) (Databricks, 2026).

**Setup:** Add your API keys as Colab secrets — `TOGETHER_API_KEY` ([get one](https://api.together.ai)) and `ANTHROPIC_API_KEY` ([get one](https://platform.claude.com/settings/keys)). Then hit **Runtime > Run all** and watch.

---

In [ ]:
!pip install -qU "konash @ git+https://github.com/konaequity/konash.git"

In [ ]:
import os, json, time, re, urllib.request, urllib.parse
from IPython.display import display, HTML

# --- Together AI Key (for KONASH agent) ---
API_KEY = None

try:
    from google.colab import userdata
    API_KEY = userdata.get("TOGETHER_API_KEY")
    if API_KEY:
        print("Together AI key loaded from Colab secrets.")
except Exception:
    pass

if not API_KEY:
    API_KEY = os.environ.get("TOGETHER_API_KEY", "")
    if API_KEY:
        print("Together AI key loaded from environment.")

if not API_KEY:
    config_path = os.path.expanduser("~/.konash/config.json")
    if os.path.exists(config_path):
        with open(config_path) as f:
            API_KEY = json.load(f).get("together_api_key", "")
        if API_KEY:
            print("Together AI key loaded from ~/.konash/config.json")

if not API_KEY:
    raise RuntimeError(
        "No TOGETHER_API_KEY found.\n"
        "  Colab: Add it as a secret (key icon in left sidebar)\n"
        "  Local: export TOGETHER_API_KEY=your-key-here"
    )

# --- Anthropic Key (for Opus 4.6 opponent) ---
ANTHROPIC_KEY = None

try:
    from google.colab import userdata
    ANTHROPIC_KEY = userdata.get("ANTHROPIC_API_KEY")
    if ANTHROPIC_KEY:
        print("Anthropic key loaded from Colab secrets.")
except Exception:
    pass

if not ANTHROPIC_KEY:
    ANTHROPIC_KEY = os.environ.get("ANTHROPIC_API_KEY", "")
    if ANTHROPIC_KEY:
        print("Anthropic key loaded from environment.")

if ANTHROPIC_KEY:
    print("\n  Head-to-head: KONASH Agent vs. Claude Opus 4.6")
else:
    print("\n  No ANTHROPIC_API_KEY — falling back to GLM 4.5 Air (no tools) as opponent.")
    print("  Add an Anthropic key for the full Opus 4.6 showdown!")

## 1. Build the Knowledge Base

We fetch 10 obscure Wikipedia articles — topics so niche that no AI model has them memorized. Both contestants will have to **search** this corpus to answer. No cheating from training data.

In [ ]:
def fetch_wikipedia(title):
    """Fetch plain-text extract of a Wikipedia article."""
    params = urllib.parse.urlencode({
        "action": "query", "titles": title, "prop": "extracts",
        "explaintext": "1", "exsectionformat": "plain", "format": "json",
    })
    url = f"https://en.wikipedia.org/w/api.php?{params}"
    req = urllib.request.Request(url, headers={"User-Agent": "KONASH-Trivia/1.0"})
    with urllib.request.urlopen(req, timeout=30) as resp:
        data = json.loads(resp.read())
    for page in data.get("query", {}).get("pages", {}).values():
        return page.get("extract", "")
    return ""


# Obscure topics — niche enough that no model has these memorized.
TOPICS = [
    "Mosquito Bowl",
    "Elinor Barker",
    "Apollon Karelin",
    "Annamite striped rabbit",
    "Harold Orlob",
    "Populus Denver",
    "2020 Hampton County tornado",
    "Dalmatius",
    "Halloween Martin",
    "High Bridge (New York City)",
]

CORPUS_DIR = "./trivia_corpus"
os.makedirs(CORPUS_DIR, exist_ok=True)

t0 = time.time()
for i, topic in enumerate(TOPICS):
    text = fetch_wikipedia(topic)
    if text and len(text) > 200:
        words = text.split()
        if len(words) > 5000:
            text = " ".join(words[:5000])
        fname = topic.replace(" ", "_").replace("/", "_").replace("(", "").replace(")", "") + ".txt"
        with open(os.path.join(CORPUS_DIR, fname), "w") as f:
            f.write(text)
        print(f"  [{i+1}/{len(TOPICS)}] {topic}: {len(text):,} chars")

print(f"\nCorpus built in {time.time()-t0:.1f}s")

## 2. Meet the Contestants

| | KONASH Agent | Claude Opus 4.6 |
|---|---|---|
| **Model** | GLM 4.5 Air | Claude Opus 4.6 |
| **Cost** | ~$0.002/query | ~$0.10/query |
| **Tools** | Corpus search | Corpus search |
| **Edge** | Trained to search | Raw intelligence |

Same questions. Same tools. Same corpus. The only difference is the model — and the price tag.

In [ ]:
import konash

agent = konash.Agent.from_preset(
    "glm-4.5-air-together",
    corpus=CORPUS_DIR,
    project="trivia-night",
    api_key=API_KEY,
)

t0 = time.time()
agent.corpus.ingest()
print(f"KONASH Agent ready: {agent.corpus.num_documents} chunks indexed in {time.time()-t0:.1f}s")

# Verify Opus connection
OPUS_MODEL = "claude-opus-4-6-20250619"
OPPONENT = "Claude Opus 4.6"

if ANTHROPIC_KEY:
    try:
        _test = json.dumps({
            "model": OPUS_MODEL, "max_tokens": 10,
            "messages": [{"role": "user", "content": "Say 'ready'."}],
        }).encode()
        _req = urllib.request.Request(
            "https://api.anthropic.com/v1/messages", data=_test,
            headers={
                "x-api-key": ANTHROPIC_KEY,
                "anthropic-version": "2023-06-01",
                "content-type": "application/json",
            },
        )
        with urllib.request.urlopen(_req, timeout=30) as _resp:
            _ = json.loads(_resp.read())
        print(f"{OPPONENT} connected.")
    except Exception as e:
        print(f"Opus connection failed ({e}) — falling back to GLM 4.5 Air (no tools)")
        ANTHROPIC_KEY = None
        OPPONENT = "GLM 4.5 Air (no tools)"
else:
    OPPONENT = "GLM 4.5 Air (no tools)"

print(f"\n  KONASH Agent  vs.  {OPPONENT}  — let's go!")

In [ ]:
import random

# ---------------------------------------------------------------------------
# Scoring
# ---------------------------------------------------------------------------

scoreboard = {"agent": 0, "opponent": 0, "questions": 0}

SEARCH_TOOL_SCHEMA = {
    "name": "search",
    "description": "Search the knowledge base for relevant documents.",
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {"type": "string", "description": "The search query."}
        },
        "required": ["query"],
    },
}


def _check_answer(response, keywords):
    """True if response contains ALL keywords (case-insensitive)."""
    text = response.lower()
    return all(kw.lower() in text for kw in keywords)


def _do_search(query, top_k=5):
    """Execute a corpus search and return formatted results."""
    results = agent.corpus.search(query, top_k=top_k)
    text = "\n\n".join(
        f"[{i+1}] (score: {r.get('score',0):.3f}) {r.get('text','')}"
        for i, r in enumerate(results)
    )
    return text, results[:3]


# ---------------------------------------------------------------------------
# Opponent: Opus 4.6 with tool use, or GLM 4.5 Air fallback
# ---------------------------------------------------------------------------

def _opponent_answer(question, max_steps=8, top_k=5):
    """Run the opponent WITH the same search tool. Returns (answer, search_log)."""
    search_log = []

    if ANTHROPIC_KEY:
        # Opus 4.6 with tool use via Anthropic Messages API
        messages = [{"role": "user", "content": (
            "Answer this trivia question using the search tool to find "
            "specific facts in the knowledge base. Be concise and cite "
            "exact names, numbers, and dates.\n\n" + question
        )}]

        for _ in range(max_steps):
            payload = json.dumps({
                "model": OPUS_MODEL,
                "max_tokens": 1024,
                "tools": [SEARCH_TOOL_SCHEMA],
                "messages": messages,
            }).encode()
            req = urllib.request.Request(
                "https://api.anthropic.com/v1/messages",
                data=payload,
                headers={
                    "x-api-key": ANTHROPIC_KEY,
                    "anthropic-version": "2023-06-01",
                    "content-type": "application/json",
                },
            )
            with urllib.request.urlopen(req, timeout=60) as resp:
                data = json.loads(resp.read())

            content_blocks = data["content"]
            messages.append({"role": "assistant", "content": content_blocks})

            # If model is done (no tool use), extract final text
            if data.get("stop_reason") != "tool_use":
                texts = [b["text"] for b in content_blocks if b.get("type") == "text"]
                return "\n".join(texts), search_log

            # Execute tool calls
            tool_results = []
            for block in content_blocks:
                if block.get("type") != "tool_use":
                    continue
                query = block.get("input", {}).get("query", "")
                result_text, top_results = _do_search(query, top_k=top_k)
                search_log.append({"query": query, "results": top_results})
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block["id"],
                    "content": result_text,
                })
            messages.append({"role": "user", "content": tool_results})

        # Max steps — return whatever text we have
        texts = [b["text"] for b in data.get("content", []) if b.get("type") == "text"]
        return ("\n".join(texts) if texts else "[Max steps reached]"), search_log

    else:
        # Fallback: GLM 4.5 Air without tools
        client = agent._get_llm_client()
        msgs = [
            {"role": "system", "content": "Answer concisely with exact names, numbers, dates."},
            {"role": "user", "content": question},
        ]
        resp = client.generate(msgs, max_tokens=400)
        text = resp.get("content", "")
        text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()
        text = re.sub(r'<think>.*', '', text, flags=re.DOTALL).strip()
        return text, []


# ---------------------------------------------------------------------------
# Scoreboard renderer
# ---------------------------------------------------------------------------

def _render_scoreboard():
    a, o = scoreboard["agent"], scoreboard["opponent"]
    total = scoreboard["questions"]
    a_color = "#a6e3a1" if a >= o else "#cdd6f4"
    o_color = "#a6e3a1" if o > a else "#cdd6f4"
    return f'''<div style="background:#1e1e2e;border-radius:10px;padding:14px 22px;
        margin:14px 0;font-family:'Fira Code','Courier New',monospace;
        display:flex;justify-content:space-around;align-items:center;
        box-shadow:0 2px 8px rgba(0,0,0,0.3);">
        <div style="text-align:center;">
            <div style="color:#89b4fa;font-size:10px;text-transform:uppercase;
                letter-spacing:1px;">KONASH Agent</div>
            <div style="color:{a_color};font-size:32px;font-weight:700;">{a}</div>
        </div>
        <div style="color:#6c7086;font-size:14px;">
            {total} question{"s" if total != 1 else ""} played</div>
        <div style="text-align:center;">
            <div style="color:#f38ba8;font-size:10px;text-transform:uppercase;
                letter-spacing:1px;">{OPPONENT}</div>
            <div style="color:{o_color};font-size:32px;font-weight:700;">{o}</div>
        </div>
    </div>'''


# ---------------------------------------------------------------------------
# Search trace renderer
# ---------------------------------------------------------------------------

def _render_trace(search_log, color, label, show=True):
    """Render search steps as HTML."""
    if not show or not search_log:
        return ""
    parts = []
    for i, entry in enumerate(search_log):
        q = entry["query"]
        snippet, source = "", ""
        if entry["results"]:
            raw = entry["results"][0].get("text", "")[:250]
            snippet = raw.replace("<", "&lt;").replace(">", "&gt;").replace('"', "&quot;")
            source = entry["results"][0].get("source", "").split("/")[-1]
        parts.append(f'''<div style="border-left:3px solid {color};
            padding:8px 12px;margin:4px 0 0 0;background:#181825;
            border-radius:0 6px 6px 0;font-size:12px;">
            <span style="color:{color};font-weight:700;">Step {i+1}</span>
            <span style="color:#6c7086;"> SEARCH</span>
            <span style="color:#89dceb;margin-left:8px;">"{q}"</span>
            <div style="color:#6c7086;font-size:11px;margin-top:2px;">
                {source}: "{snippet[:200]}..."</div></div>''')
    header = f'<div style="margin:8px 0 4px 0;color:#6c7086;font-size:11px;">{label} search trace ({len(search_log)} queries):</div>'
    return header + "".join(parts)


# ---------------------------------------------------------------------------
# Main trivia function
# ---------------------------------------------------------------------------

def trivia_question(agent, question, keywords, points=1, category="General",
                    max_steps=10, top_k=5, show_trace=True):
    """Run one trivia question: both contestants search the corpus, score both."""
    from konash.harness.environment import Environment
    from konash.plugins.compression import CompressionPlugin
    from konash.plugins.control import StepBudgetPlugin

    if not agent.corpus.indexed:
        agent.corpus.ingest()

    q_escaped = question.replace("<", "&lt;").replace(">", "&gt;")
    ref_display = ", ".join(keywords)

    # --- Opponent (Opus 4.6 with search) ---
    t0 = time.time()
    try:
        opponent_text, opponent_searches = _opponent_answer(question, max_steps=max_steps, top_k=top_k)
    except Exception as e:
        opponent_text, opponent_searches = f"[Error: {e}]", []
    opponent_time = time.time() - t0
    opponent_correct = _check_answer(opponent_text, keywords)

    # --- KONASH Agent (with corpus search) ---
    base_agent = agent._make_agent(max_steps=max_steps)
    agent_searches = []

    def tool_executor(tc):
        q = ""
        if isinstance(tc, dict):
            fn = tc.get("function", {})
            if isinstance(fn, dict):
                args = fn.get("arguments", "{}")
                if isinstance(args, str):
                    try:
                        args = json.loads(args)
                    except Exception:
                        args = {}
                if isinstance(args, dict):
                    q = args.get("query", "") or args.get("input", "")
            if not q:
                q = tc.get("query", "") or tc.get("input", "")
        if not q:
            q = str(tc)
        result_text, top_results = _do_search(q, top_k=top_k)
        agent_searches.append({"query": q, "results": top_results})
        obs = {"role": "tool", "content": result_text}
        if isinstance(tc, dict) and tc.get("id"):
            obs["tool_call_id"] = tc["id"]
        return obs

    env = Environment(
        tool_executor=tool_executor,
        plugins=[
            CompressionPlugin(threshold_tokens=8000, target_tokens=4000),
            StepBudgetPlugin(max_steps=max_steps),
        ],
        available_tools=[{
            "type": "function",
            "function": {
                "name": "search",
                "description": "Search the knowledge base for relevant documents.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "query": {"type": "string", "description": "The search query."}
                    },
                    "required": ["query"],
                },
            },
        }],
    )

    env.reset(prompt=question)
    t1 = time.time()
    for step in range(max_steps):
        result = env.step(base_agent)
        if result["done"]:
            break
    agent_time = time.time() - t1

    agent_answer = base_agent.extract_final_answer(env.conversation_history) or ""
    agent_answer = re.sub(r'<think>.*?</think>', '', agent_answer, flags=re.DOTALL).strip()
    agent_answer = re.sub(r'<think>.*', '', agent_answer, flags=re.DOTALL).strip()
    agent_correct = _check_answer(agent_answer, keywords)

    # --- Score ---
    if agent_correct:
        scoreboard["agent"] += points
    if opponent_correct:
        scoreboard["opponent"] += points
    scoreboard["questions"] += 1

    # --- Render ---
    opp_escaped = opponent_text[:500].replace("<", "&lt;").replace(">", "&gt;")
    agent_escaped = agent_answer[:600].replace("<", "&lt;").replace(">", "&gt;")
    o_badge = (f'<span style="color:#a6e3a1;font-weight:700;">CORRECT +{points}</span>'
               if opponent_correct else
               '<span style="color:#f38ba8;font-weight:700;">WRONG</span>')
    a_badge = (f'<span style="color:#a6e3a1;font-weight:700;">CORRECT +{points}</span>'
               if agent_correct else
               '<span style="color:#f38ba8;font-weight:700;">WRONG</span>')

    html = f'''<div style="background:#1e1e2e;border-radius:10px;padding:18px 22px;
        margin:14px 0;font-family:'Fira Code','Courier New',monospace;font-size:13px;
        line-height:1.6;color:#cdd6f4;box-shadow:0 2px 8px rgba(0,0,0,0.3);">
        <div style="display:flex;justify-content:space-between;align-items:center;">
            <div style="color:#cba6f7;font-size:10px;text-transform:uppercase;
                letter-spacing:1.5px;font-weight:700;">{category}</div>
            <div style="color:#f9e2af;font-size:11px;">{points} point{"s" if points > 1 else ""}</div>
        </div>
        <div style="font-size:15px;margin:6px 0 14px 0;">{q_escaped}</div>'''

    # Opponent search trace + answer
    html += _render_trace(opponent_searches, "#f38ba8", OPPONENT, show=show_trace)
    html += f'''<div style="border-left:3px solid #f38ba8;padding:10px 14px;
        margin:6px 0;background:#2a1a1a;border-radius:0 6px 6px 0;">
        <div style="display:flex;justify-content:space-between;">
            <span style="color:#f38ba8;font-weight:700;font-size:12px;">
                {OPPONENT}</span>
            <span style="font-size:11px;">{o_badge} &middot; {opponent_time:.1f}s &middot;
                {len(opponent_searches)} searches</span>
        </div>
        <div style="color:#bac2de;margin:4px 0;font-size:12px;">
            {opp_escaped}</div></div>'''

    # Agent search trace + answer
    html += _render_trace(agent_searches, "#89b4fa", "KONASH Agent", show=show_trace)
    html += f'''<div style="border-left:3px solid #89b4fa;padding:10px 14px;
        margin:6px 0;background:#1a1a2a;border-radius:0 6px 6px 0;">
        <div style="display:flex;justify-content:space-between;">
            <span style="color:#89b4fa;font-weight:700;font-size:12px;">
                KONASH Agent</span>
            <span style="font-size:11px;">{a_badge} &middot; {agent_time:.1f}s &middot;
                {len(agent_searches)} searches</span>
        </div>
        <div style="color:#cdd6f4;margin:4px 0;font-size:12px;">
            {agent_escaped}</div></div>'''

    # Key answer
    html += f'''<div style="color:#6c7086;font-size:11px;margin-top:10px;
        padding-top:8px;border-top:1px solid #313244;">
        Key: {ref_display}</div></div>'''

    display(HTML(html))
    display(HTML(_render_scoreboard()))


print("Trivia Night ready!")

## 3. Round 1 — Warm-Up

**1 point each.** These facts are buried in obscure Wikipedia articles. Neither model has them memorized — they have to search.

In [ ]:
# Round 1: Warm-Up — 1 point each
# Every question here was verified: Opus 4.6 cannot answer from memory.
ROUND_1 = [
    {
        # Opus knows Guadalcanal but NOT the final score.
        "question": "The Mosquito Bowl was a WWII football game between Marines on Guadalcanal. What was the final score?",
        "keywords": ["0-0"],
        "category": "WWII History",
    },
    {
        # Opus punted on this entirely.
        "question": "The Populus Denver hotel has a distinctive facade. What common kitchen item has it been compared to?",
        "keywords": ["cheese grater"],
        "category": "Architecture",
    },
    {
        # Opus guessed "pregnant" but won't know this specific detail.
        "question": "Olympic champion cyclist Elinor Barker joined a cycling club at age ten to avoid swimming classes. What was the club called?",
        "keywords": ["Maindy Flyers"],
        "category": "Sports",
    },
]

for q in ROUND_1:
    trivia_question(agent, q["question"], q["keywords"],
                    points=1, category=q["category"], max_steps=10)

## 4. Round 2 — Deep Cuts

**2 points each.** Multi-part questions with specific names, dates, and numbers. Pure search skill.

In [ ]:
# Round 2: Deep Cuts — 2 points each
# Multi-part questions with extremely specific details.
ROUND_2 = [
    {
        # Opus knows the Knights Templar connection but NOT this detail.
        "question": (
            "Russian anarchist Apollon Karelin spent 12 years in exile. "
            "What was his nickname during his time in Paris?"
        ),
        "keywords": ["Beard"],
        "category": "Political History",
    },
    {
        # Opus was vague ("lawsuit", "1940s") — not specific enough.
        "question": (
            "Harold Orlob composed the melody for the 1909 hit song "
            "'I Wonder Who's Kissing Her Now' but was not credited. "
            "How did he eventually receive credit, and in what year?"
        ),
        "keywords": ["sued", "1947"],
        "category": "Music History",
    },
    {
        # Opus knows Dalmatius was killed, but NOT where he was made Caesar.
        "question": (
            "At what specific location was Dalmatius raised to the "
            "rank of Caesar on 18 September 335 AD?"
        ),
        "keywords": ["Ripa Gothica"],
        "category": "Ancient Rome",
    },
]

for q in ROUND_2:
    trivia_question(agent, q["question"], q["keywords"],
                    points=2, category=q["category"], max_steps=12)

## 5. Round 3 — Lightning Round

Quick-fire. **1 point each.** No search traces shown — just answers and scores.

In [ ]:
# Round 3: Lightning Round — 1 point each, no trace, fast
ROUND_3 = [
    {
        # Opus punted entirely.
        "question": "The 2020 Hampton County tornado set a historic record for South Carolina. What was its Enhanced Fujita rating?",
        "keywords": ["EF4"],
        "category": "Weather",
    },
    {
        # Opus punted entirely.
        "question": "What was Halloween Martin's pioneering role in the history of American radio?",
        "keywords": ["disc jockey"],
        "category": "Media History",
    },
    {
        # Opus knows "market" but NOT the ancestor species.
        "question": "What extinct species, whose fossils were found in Guangxi, China, is the presumed ancestor of the Annamite striped rabbit?",
        "keywords": ["sinensis"],
        "category": "Zoology",
    },
    {
        # Impossibly obscure — an 1860s industrial supplier.
        "question": "Which company supplied the metal for the 90-inch water pipe added to New York City's High Bridge in the early 1860s?",
        "keywords": ["Abbott"],
        "category": "NYC Trivia",
    },
]

for q in ROUND_3:
    trivia_question(agent, q["question"], q["keywords"],
                    points=1, category=q["category"], max_steps=8,
                    show_trace=False)

## 🏆 Final Score

In [ ]:
# Final results
a, o = scoreboard["agent"], scoreboard["opponent"]
total = scoreboard["questions"]
max_pts = sum(q.get("points", 1) for q in ROUND_1) + sum(q.get("points", 2) for q in ROUND_2) + sum(q.get("points", 1) for q in ROUND_3)

if a > o:
    verdict = "KONASH Agent wins!"
    verdict_color = "#89b4fa"
    subtitle = "Same tools, same corpus — the trained agent comes out on top."
elif o > a:
    verdict = f"{OPPONENT} wins!"
    verdict_color = "#f38ba8"
    subtitle = "Raw intelligence prevails. But compare the cost below..."
else:
    verdict = "It's a tie!"
    verdict_color = "#f9e2af"
    subtitle = "Matching the world's best model — at a fraction of the cost."

display(HTML(f'''<div style="background:#1e1e2e;border-radius:12px;padding:24px 30px;
    margin:14px 0;font-family:'Fira Code','Courier New',monospace;
    text-align:center;box-shadow:0 4px 16px rgba(0,0,0,0.4);">
    <div style="font-size:28px;font-weight:700;color:{verdict_color};
        margin-bottom:6px;">{verdict}</div>
    <div style="color:#6c7086;font-size:12px;margin-bottom:16px;">{subtitle}</div>
    <div style="display:flex;justify-content:center;gap:60px;margin:16px 0;">
        <div>
            <div style="color:#89b4fa;font-size:11px;text-transform:uppercase;
                letter-spacing:1px;">KONASH Agent</div>
            <div style="color:{"#a6e3a1" if a >= o else "#cdd6f4"};font-size:48px;
                font-weight:700;">{a}</div>
            <div style="color:#6c7086;font-size:12px;">of {max_pts} possible</div>
            <div style="color:#89b4fa;font-size:10px;margin-top:4px;">GLM 4.5 Air &middot; ~$0.002/query</div>
        </div>
        <div>
            <div style="color:#f38ba8;font-size:11px;text-transform:uppercase;
                letter-spacing:1px;">{OPPONENT}</div>
            <div style="color:{"#a6e3a1" if o > a else "#cdd6f4"};font-size:48px;
                font-weight:700;">{o}</div>
            <div style="color:#6c7086;font-size:12px;">of {max_pts} possible</div>
            <div style="color:#f38ba8;font-size:10px;margin-top:4px;">Opus 4.6 &middot; ~$0.10/query</div>
        </div>
    </div>
    <div style="color:#6c7086;font-size:12px;margin-top:12px;padding-top:12px;
        border-top:1px solid #313244;">
        {total} questions &middot; same tools &middot; same corpus &middot;
        50x cost difference</div>
</div>'''))

## 7. Train with OAPL (Optional — requires GPU)

KONASH trains agents with **OAPL** (Off-policy Advantage-weighted Policy Learning): synthesize hard QA pairs from the corpus, generate search-and-answer rollouts, filter to the learning frontier, and update the model with RL. This is what turns base GLM 4.5 Air into a KARL-style knowledge agent.

Skip this cell if running without a GPU — the agent above already works for inference.

In [ ]:
# Check for GPU — training requires CUDA for gradient updates
try:
    import torch
    HAS_GPU = torch.cuda.is_available()
except ImportError:
    HAS_GPU = False

if HAS_GPU:
    from konash.synthesis.qa import SyntheticExample

    # Split mode: Together AI for fast inference, local Qwen3-4B for OAPL gradients
    train_agent = konash.Agent(
        base_model="Qwen/Qwen3-4B",
        corpus=agent.corpus,
        project="trivia-night",
        load_in_4bit=True,
        gradient_checkpointing=True,
        inference_api_base="https://api.together.xyz/v1",
        inference_api_key=API_KEY,
        inference_model="zai-org/GLM-4.5-Air-FP8",
    )

    FEW_SHOT = [
        SyntheticExample(
            question="The Mosquito Bowl was a football game played between Marines "
            "during WWII. On which island was it played, and in what year?",
            answer="It was played on Guadalcanal on December 24, 1944.",
        ),
        SyntheticExample(
            question="What medieval order did Russian anarchist Apollon Karelin "
            "establish to promote his ideology?",
            answer="He established an Order of the Knights Templar to promote anarchism.",
        ),
        SyntheticExample(
            question="Harold Orlob was the uncredited composer of a 1909 hit song. "
            "What was the song and how did he finally get credit?",
            answer="The song was 'I Wonder Who's Kissing Her Now'. He sued in 1947 "
            "and was finally credited as the composer.",
        ),
    ]

    print("Starting OAPL training (1 iteration)...\n")
    result = train_agent.train(
        iterations=1,
        synthesis_calls=5,
        rollouts_per_example=2,
        max_examples=5,
        few_shot_examples=FEW_SHOT,
        verbose=True,
    )
    print(f"\nTraining result: {result}")
else:
    print("No GPU detected — skipping training.")
    print("Training requires a CUDA GPU (e.g. Colab T4 runtime).")
    print("\nThe inference agent above works without a GPU.")

## 8. Your Turn

Ask your own questions about any topic in the corpus: Mosquito Bowl, Elinor Barker, Apollon Karelin, Annamite striped rabbit, Harold Orlob, Populus Denver, 2020 Hampton County tornado, Dalmatius, Halloween Martin, High Bridge (NYC).

In [ ]:
# Edit these and re-run — challenge Opus on your own question!
your_question = "Who wrote the book about the Mosquito Bowl, and when was it published?"
your_keywords = ["Bissinger", "2022"]  # keywords the answer must contain

trivia_question(agent, your_question, your_keywords,
                points=1, category="Bonus Round", max_steps=10)

---

**What you just saw:** Two AI models with the **same search tool** and the **same knowledge base** competing on hard trivia. The KONASH agent (GLM 4.5 Air) costs ~50x less per query than Claude Opus 4.6 — and holds its own.

This is the core finding of the [KARL paper](https://arxiv.org/abs/2503.xxxxx) (Databricks, 2026): a smaller model trained with RL to search effectively matches frontier models on knowledge-intensive benchmarks. KARL (par. N=20) scores **68.1** on KARLBench vs. Claude Opus 4.6's **67.5**.

**Next steps:**
- Add your own documents to the corpus for domain-specific agents
- Run OAPL training on a GPU to further improve the agent's search strategy via RL
- Try `konash setup` in your terminal for guided setup

[GitHub](https://github.com/konaequity/konash)